# Advanced First-Order Methods

| Difficulty | █████████░░░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 04.10, 04.05 |
| Optional | 10.03 (natural gradient — for the mirror descent connection) |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/12_advanced_first_order_methods.ipynb)

---

## 🎯 Learning Objective

Master three powerful first-order methods — ADMM, Frank-Wolfe, and mirror descent — that
extend the optimization toolkit beyond standard (proximal) gradient descent, and understand
when each is the right tool for large-scale ML problems.

---

## 📐 Theory

`04.10` introduced proximal operators and ISTA/FISTA for composite minimization;
`04.05` developed constrained optimization via KKT conditions and projected gradient
descent. This notebook presents three advanced first-order methods that decompose,
constrain, or reshape optimization problems in fundamentally different ways — plus a
brief survey of stochastic variance-reduction techniques that close the gap between
stochastic and deterministic convergence.

### 1 · ADMM (Alternating Direction Method of Multipliers)

Consider structured problems of the form

$$\min_{x,\,z}\; f(x) + g(z) \quad \text{subject to} \quad Ax + Bz = c$$

where $f$ and $g$ are convex (possibly non-smooth) and the constraint couples the two
blocks of variables. The **augmented Lagrangian** adds both a Lagrange multiplier $u$
(the dual variable) and a quadratic penalty with parameter $\rho > 0$:

$$\mathcal{L}_\rho(x, z, u) = f(x) + g(z) + u^\top(Ax + Bz - c)
  + \frac{\rho}{2}\|Ax + Bz - c\|_2^2$$

ADMM decomposes the joint minimization into three alternating updates (the scaled form,
absorbing $u/\rho$ into a scaled dual variable $u$):

$$\boxed{\begin{aligned}
x^{k+1} &= \arg\min_x \left[ f(x) + \frac{\rho}{2}\|Ax + Bz^k - c + u^k\|_2^2 \right]
  & \text{(x-update)} \\
z^{k+1} &= \arg\min_z \left[ g(z) + \frac{\rho}{2}\|Ax^{k+1} + Bz - c + u^k\|_2^2 \right]
  & \text{(z-update)} \\
u^{k+1} &= u^k + Ax^{k+1} + Bz^{k+1} - c
  & \text{(dual update)}
\end{aligned}}$$

Each subproblem involves only $f$ or $g$ (not both), so if each is individually easy to
minimize, ADMM converts a hard coupled problem into a sequence of easy ones. The dual
variable update accumulates the running constraint violation, steadily driving the
solution toward feasibility.

**Convergence:** $O(1/T)$ for convex $f, g$ — the primal residual $\|Ax^k + Bz^k - c\|$
and the dual residual $\|\rho B^\top(z^k - z^{k-1})\|$ both converge to zero. The penalty
parameter $\rho$ trades off primal vs. dual convergence speed; adaptive $\rho$-selection
schemes monitor both residuals and adjust.

**ML applications:**
- **Distributed optimization** — split data across $K$ machines, each solves a local
  subproblem $\min_{x_i} f_i(x_i)$, and ADMM's consensus constraint $x_i = z$ (a shared
  global variable) coordinates them without ever centralizing the data.
- **Robust PCA** — decompose a matrix $M = L + S$ where $L$ is low-rank ($\min \|L\|_*$)
  and $S$ is sparse ($\min \lambda\|S\|_1$); ADMM alternates a singular-value thresholding
  step (for $L$) with a soft-thresholding step (for $S$).
- **Consensus optimization** — enforce agreement across distributed agents while allowing
  heterogeneous local objectives.

### 2 · Frank-Wolfe (Conditional Gradient)

For constrained problems $\min_{w \in C} f(w)$ over a compact convex set $C$, the
**Frank-Wolfe** algorithm replaces the expensive projection step of projected GD with a
*linear minimization oracle* (LMO):

$$\boxed{\begin{aligned}
s_t &= \arg\min_{s \in C}\; \langle \nabla f(w_t),\, s \rangle
  & \text{(linear minimization over } C\text{)} \\
w_{t+1} &= w_t + \gamma_t(s_t - w_t)
  & \text{(convex combination step, } \gamma_t = \tfrac{2}{t+2}\text{)}
\end{aligned}}$$

Since $w_{t+1}$ is a convex combination of $w_t \in C$ and $s_t \in C$, **every iterate
stays feasible** — no projection needed. After $T$ steps, $w_T$ is a convex combination
of at most $T$ vertices of $C$, producing **structurally sparse iterates**.

**Key insight — sparsity from geometry:** For the $\ell_1$-ball
$C = \{w : \|w\|_1 \le r\}$, the LMO returns $s_t = -r\,\text{sign}(\nabla f(w_t)_j)\,e_j$
where $j = \arg\max_i |\nabla f(w_t)_i|$ — a single coordinate direction. For the
**nuclear-norm ball** $\{W : \|W\|_* \le r\}$, the LMO returns $r\,u_1 v_1^\top$
(a rank-1 matrix from the leading singular vectors), so Frank-Wolfe incrementally builds
a low-rank solution — one rank at a time.

**Convergence:** $O(1/T)$ — specifically, with the curvature constant
$C_f = \sup_{x,s \in C,\, \gamma \in [0,1]} \frac{2}{\gamma^2}[f(x + \gamma(s-x))
- f(x) - \gamma\langle \nabla f(x), s - x\rangle]$, we have
$f(w_T) - f^* \le \frac{2C_f}{T+2}$. This cannot be accelerated to $O(1/T^2)$ in
general — projection-free optimization pays a price in rate, but gains enormously in
per-iteration cost when projection is expensive and linear minimization is cheap.

### 3 · Mirror Descent

Projected gradient descent measures "closeness" between iterates using the Euclidean
distance $\frac{1}{2}\|w - v\|_2^2$. **Mirror descent** generalizes this by replacing
Euclidean distance with a **Bregman divergence** $D_\phi(w, v)$ derived from a strictly
convex **mirror map** $\phi$:

$$D_\phi(w, v) = \phi(w) - \phi(v) - \nabla\phi(v)^\top(w - v) \ge 0$$

The mirror descent update:

$$\boxed{w_{t+1} = \arg\min_{w \in C}\left\{
  \langle g_t,\, w \rangle + \frac{1}{\eta_t} D_\phi(w, w_t) \right\}}$$

where $g_t \in \partial f(w_t)$ is a (sub)gradient.

**Two fundamental instantiations:**

| Mirror map $\phi(w)$ | Bregman divergence | Algorithm recovered |
| :--- | :--- | :--- |
| $\frac{1}{2}\|w\|_2^2$ (Euclidean) | $\frac{1}{2}\|w-v\|_2^2$ | **Projected gradient descent** (`04.05`) |
| $\sum_j w_j \log w_j$ (negative entropy) | $\text{KL}(w \| v) = \sum_j w_j \log(w_j/v_j)$ | **Exponentiated gradient** (multiplicative weights) |

The Euclidean mirror map gives standard projected GD — so projected GD is a *special case*
of mirror descent. The **negative-entropy mirror map** on the probability simplex
$\Delta = \{w \ge 0,\, \sum w_j = 1\}$ produces the exponentiated gradient update:

$$w_{t+1,j} = \frac{w_{t,j}\, \exp(-\eta_t\, g_{t,j})}{\sum_i w_{t,i}\,
  \exp(-\eta_t\, g_{t,i})}$$

which is multiplicative (not additive) and naturally stays on the simplex — no projection
needed. This is the geometry-aware choice for distributions: the KL divergence respects
the simplex's intrinsic structure, while Euclidean projection distorts it.

**Connection to natural gradient:** The Fisher information matrix $F(\theta)$ defines a
Riemannian metric on the parameter space of probability distributions. The natural
gradient update $\theta_{t+1} = \theta_t - \eta F(\theta_t)^{-1}\nabla f(\theta_t)$
(`04.07`, `10.03`) can be interpreted as mirror descent with the KL divergence as the
Bregman divergence — both adapt the notion of "distance" to the geometry of the problem
rather than imposing Euclidean structure.

### 4 · Stochastic Variance Reduction (Brief Survey)

For finite-sum problems $f(w) = \frac{1}{n}\sum_{i=1}^n f_i(w)$ (the standard ML setting),
vanilla SGD achieves $O(1/\sqrt{T})$ for convex objectives — the variance of the stochastic
gradient prevents faster convergence. **Variance-reduction** methods close this gap:

- **SVRG** (Stochastic Variance-Reduced Gradient, Johnson & Zhang 2013): periodically
  computes a full gradient $\nabla f(\tilde{w})$ at a snapshot point, then uses the
  *corrected* stochastic gradient $\nabla f_i(w_t) - \nabla f_i(\tilde{w}) + \nabla
  f(\tilde{w})$ which has zero variance at $\tilde{w}$ and vanishing variance as
  $w_t \to w^*$.
- **SAGA** (Defazio et al. 2014): maintains a table of per-sample gradients and uses a
  similar correction without requiring periodic full-gradient passes.

Both achieve **linear convergence** ($O(\rho^T)$) for strongly convex finite-sum problems
— matching the full-batch rate with stochastic per-iteration cost. This explains why SGD
on finite datasets empirically converges much faster than the $O(1/\sqrt{T})$ worst-case
rate would predict.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(42)

## 👁️ Visual Intuition

Three panels below expose the geometric character of each method:

- **ADMM** — we visualize convergence of the primal and dual residuals on a Lasso
  splitting, showing how the two subproblems alternately reduce constraint violation.
- **Frank-Wolfe** — iterates on the $\ell_1$ ball, showing how each step moves toward
  a single vertex (coordinate direction) and the solution remains a sparse convex
  combination of vertices.
- **Mirror descent** — Euclidean projected GD vs. entropy-mirror (exponentiated gradient)
  on the 2-simplex, showing how the choice of geometry fundamentally changes the
  optimization trajectory.

In [ ]:
# ---------- Panel 1: ADMM convergence (primal & dual residuals) ----------
np.random.seed(42)
n, d = 80, 120
X = np.random.randn(n, d)
w_true = np.zeros(d); w_true[:8] = np.random.randn(8) * 3
y = X @ w_true + 0.1 * np.random.randn(n)
lam = 0.5

def soft_thresh(v, t):
    return np.sign(v) * np.maximum(np.abs(v) - t, 0)

# ADMM for Lasso: min 0.5||Xw-y||^2 + lam||z||_1  s.t. w = z
rho_admm = 1.0
XtX = X.T @ X
Xty = X.T @ y
L_factor = np.linalg.cholesky(XtX + rho_admm * np.eye(d))

w_admm = np.zeros(d); z_admm = np.zeros(d); u_admm = np.zeros(d)
primal_res, dual_res = [], []
for _ in range(100):
    # x-update: (X^TX + rho*I) w = X^Ty + rho*(z - u)
    rhs = Xty + rho_admm * (z_admm - u_admm)
    w_admm = np.linalg.solve(L_factor @ L_factor.T, rhs)
    z_old = z_admm.copy()
    # z-update: soft-thresholding
    z_admm = soft_thresh(w_admm + u_admm, lam / rho_admm)
    # dual update
    u_admm = u_admm + w_admm - z_admm
    primal_res.append(np.linalg.norm(w_admm - z_admm))
    dual_res.append(rho_admm * np.linalg.norm(z_admm - z_old))

# ---------- Panel 2: Frank-Wolfe on L1 ball ----------
# min f(w) = 0.5*w^T Q w - q^T w  s.t. ||w||_1 <= 1   (2D for visualization)
Q_fw = np.array([[3.0, 1.0], [1.0, 2.0]])
q_fw = np.array([1.0, 0.5])

w_fw = np.array([0.0, 0.0])
fw_path = [w_fw.copy()]
fw_gaps = []
for t in range(30):
    grad = Q_fw @ w_fw - q_fw
    # LMO over L1 ball: s = -sign(grad_j) * e_j where j = argmax |grad|
    j = np.argmax(np.abs(grad))
    s = np.zeros(2); s[j] = -np.sign(grad[j])
    gamma = 2.0 / (t + 2)
    w_fw = w_fw + gamma * (s - w_fw)
    fw_path.append(w_fw.copy())
    fw_gaps.append(0.5 * w_fw @ Q_fw @ w_fw - q_fw @ w_fw)
fw_path = np.array(fw_path)

# ---------- Panel 3: Mirror descent comparison on simplex ----------
# min f(w) = 0.5*w^T A w - b^T w  on simplex {w >= 0, sum = 1}
A_md = np.array([[4.0, 1.0, 0.5], [1.0, 3.0, 0.5], [0.5, 0.5, 2.0]])
b_md = np.array([0.2, 0.8, 0.3])

def proj_simplex(v):
    """Project onto probability simplex (Duchi et al. 2008)."""
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u) - 1
    rho_idx = np.nonzero(u * np.arange(1, len(v)+1) > cssv)[0][-1]
    theta = cssv[rho_idx] / (rho_idx + 1.0)
    return np.maximum(v - theta, 0)

eta_md = 0.15
# Projected GD (Euclidean mirror)
w_pgd = np.array([1/3, 1/3, 1/3])
pgd_path = [w_pgd.copy()]
for _ in range(40):
    g = A_md @ w_pgd - b_md
    w_pgd = proj_simplex(w_pgd - eta_md * g)
    pgd_path.append(w_pgd.copy())
pgd_path = np.array(pgd_path)

# Exponentiated gradient (entropy mirror)
w_eg = np.array([1/3, 1/3, 1/3])
eg_path = [w_eg.copy()]
for _ in range(40):
    g = A_md @ w_eg - b_md
    w_eg = w_eg * np.exp(-eta_md * g)
    w_eg /= w_eg.sum()
    eg_path.append(w_eg.copy())
eg_path = np.array(eg_path)

# ---------- Plotting ----------
fig = plt.figure(figsize=(16, 4.5))
gs = GridSpec(1, 3, figure=fig, wspace=0.32)

# Panel 1: ADMM residuals
ax1 = fig.add_subplot(gs[0, 0])
ax1.semilogy(primal_res, label='primal ||w - z||', color='#4C72B0', lw=2)
ax1.semilogy(dual_res, label='dual ρ||Δz||', color='#C44E52', lw=2, ls='--')
ax1.set_xlabel('ADMM iteration'); ax1.set_ylabel('residual (log)')
ax1.set_title('ADMM convergence on Lasso\n(primal & dual residuals → 0)')
ax1.legend(fontsize=8)

# Panel 2: Frank-Wolfe on L1 ball
ax2 = fig.add_subplot(gs[0, 1])
# Draw L1 ball boundary
l1_verts = np.array([[1,0],[0,1],[-1,0],[0,-1],[1,0]])
ax2.plot(l1_verts[:,0], l1_verts[:,1], 'k-', lw=1, alpha=0.4)
ax2.fill(l1_verts[:,0], l1_verts[:,1], alpha=0.07, color='gray')
# Contours of f
xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 100), np.linspace(-1.2, 1.2, 100))
zz = np.array([0.5 * np.array([x,y]) @ Q_fw @ np.array([x,y]) - q_fw @ np.array([x,y])
               for x, y in zip(xx.ravel(), yy.ravel())]).reshape(xx.shape)
ax2.contour(xx, yy, zz, levels=15, alpha=0.3, colors='#4C72B0')
ax2.plot(fw_path[:,0], fw_path[:,1], 'o-', color='#C44E52', ms=3, lw=1.5,
         label='FW iterates')
ax2.set_title('Frank-Wolfe on ℓ₁ ball\n(sparse vertex-chasing iterates)')
ax2.set_xlim(-1.3, 1.3); ax2.set_ylim(-1.3, 1.3)
ax2.set_aspect('equal'); ax2.legend(fontsize=8)

# Panel 3: Mirror descent on simplex (plot w1 vs w2, since w3 = 1 - w1 - w2)
ax3 = fig.add_subplot(gs[0, 2])
# Simplex boundary in (w1, w2) space
tri = np.array([[0,0],[1,0],[0,1],[0,0]])
ax3.plot(tri[:,0], tri[:,1], 'k-', lw=1, alpha=0.4)
ax3.fill(tri[:,0], tri[:,1], alpha=0.07, color='gray')
ax3.plot(pgd_path[:,0], pgd_path[:,1], 'o-', color='#4C72B0', ms=3, lw=1.5,
         label='Projected GD (Euclidean)', alpha=0.8)
ax3.plot(eg_path[:,0], eg_path[:,1], 's-', color='#55A868', ms=3, lw=1.5,
         label='Exponentiated grad (entropy)', alpha=0.8)
ax3.set_xlabel('w₁'); ax3.set_ylabel('w₂')
ax3.set_title('Mirror descent on simplex\n(geometry changes trajectory)')
ax3.set_xlim(-0.05, 0.75); ax3.set_ylim(-0.05, 0.75)
ax3.set_aspect('equal'); ax3.legend(fontsize=7)

plt.tight_layout()
plt.show()

print(f"ADMM final residuals — primal: {primal_res[-1]:.2e}, dual: {dual_res[-1]:.2e}")
print(f"Frank-Wolfe: ||w_T||_1 = {np.abs(fw_path[-1]).sum():.4f} (≤ 1 constraint satisfied)")
print(f"FW sparsity: {np.sum(np.abs(fw_path[-1]) > 1e-10)}/{len(fw_path[-1])} nonzero coords")
print(f"Mirror descent final — PGD: {pgd_path[-1].round(4)}, EG: {eg_path[-1].round(4)}")

## 🐍 Implementation from Scratch

We implement all three methods from first principles, solving problems where each
method's strengths are most apparent:

1. **ADMM for Lasso** — the splitting $w = z$ converts the composite
   $\frac{1}{2}\|Xw - y\|^2 + \lambda\|w\|_1$ into a ridge-regression $x$-update
   (closed-form) and a soft-thresholding $z$-update. We compare convergence to ISTA/FISTA
   from `04.10`.
2. **Frank-Wolfe for a quadratic over the $\ell_1$ ball** — the LMO is a simple
   $\arg\max$ over gradient coordinates, producing sparse iterates.
3. **Entropy mirror descent on the simplex** — the exponentiated gradient update
   stays on the probability simplex by construction, and we compare its convergence
   profile against Euclidean projected GD.

In [ ]:
# ============================================================================
# 1. ADMM for Lasso: min 0.5||Xw - y||^2 + lam*||w||_1
#    Splitting: min 0.5||Xw - y||^2 + lam*||z||_1  s.t. w = z
# ============================================================================
np.random.seed(7)
n_lasso, d_lasso = 100, 200
X_lasso = np.random.randn(n_lasso, d_lasso)
w_star = np.zeros(d_lasso)
w_star[:10] = np.random.randn(10) * 2  # sparse ground truth
y_lasso = X_lasso @ w_star + 0.05 * np.random.randn(n_lasso)
lam_lasso = 0.3

def lasso_objective(w, X, y, lam):
    return 0.5 * np.sum((X @ w - y)**2) + lam * np.linalg.norm(w, 1)

# --- ADMM ---
def admm_lasso(X, y, lam, rho=1.0, max_iter=200):
    n, d = X.shape
    XtX = X.T @ X
    Xty = X.T @ y
    # Precompute Cholesky factor for x-update
    L_chol = np.linalg.cholesky(XtX + rho * np.eye(d))
    w = np.zeros(d); z = np.zeros(d); u = np.zeros(d)
    obj_hist = []
    for _ in range(max_iter):
        # x-update: solve (X^TX + rho*I) w = X^Ty + rho*(z - u)
        w = np.linalg.solve(L_chol, Xty + rho * (z - u))
        w = np.linalg.solve(L_chol.T, w)
        # z-update: soft-thresholding
        z = soft_thresh(w + u, lam / rho)
        # dual update
        u = u + w - z
        obj_hist.append(lasso_objective(z, X, y, lam))
    return z, obj_hist

# --- ISTA (from 04.10 for comparison) ---
def ista_lasso(X, y, lam, max_iter=200):
    n, d = X.shape
    L_lip = np.linalg.eigvalsh(X.T @ X).max()  # Lipschitz constant
    eta = 1.0 / L_lip
    w = np.zeros(d)
    obj_hist = []
    for _ in range(max_iter):
        grad = X.T @ (X @ w - y)
        w = soft_thresh(w - eta * grad, lam * eta)
        obj_hist.append(lasso_objective(w, X, y, lam))
    return w, obj_hist

# --- FISTA ---
def fista_lasso(X, y, lam, max_iter=200):
    n, d = X.shape
    L_lip = np.linalg.eigvalsh(X.T @ X).max()
    eta = 1.0 / L_lip
    w = np.zeros(d); w_prev = w.copy(); t_k = 1.0
    obj_hist = []
    for _ in range(max_iter):
        t_k1 = (1 + np.sqrt(1 + 4 * t_k**2)) / 2
        v = w + ((t_k - 1) / t_k1) * (w - w_prev)
        grad = X.T @ (X @ v - y)
        w_prev = w.copy()
        w = soft_thresh(v - eta * grad, lam * eta)
        t_k = t_k1
        obj_hist.append(lasso_objective(w, X, y, lam))
    return w, obj_hist

w_admm_sol, obj_admm = admm_lasso(X_lasso, y_lasso, lam_lasso)
w_ista_sol, obj_ista = ista_lasso(X_lasso, y_lasso, lam_lasso)
w_fista_sol, obj_fista = fista_lasso(X_lasso, y_lasso, lam_lasso)

# ============================================================================
# 2. Frank-Wolfe for quadratic over L1 ball
#    min 0.5*w^T Q w - q^T w   s.t. ||w||_1 <= r
# ============================================================================
d_fw = 50
Q_fw_big = np.random.randn(d_fw, d_fw)
Q_fw_big = Q_fw_big.T @ Q_fw_big / d_fw + 0.5 * np.eye(d_fw)  # PSD
q_fw_big = np.random.randn(d_fw)
r_ball = 2.0

def fw_l1(Q, q, r, max_iter=200):
    d = len(q)
    w = np.zeros(d)  # start at origin (feasible)
    obj_hist = []
    sparsity_hist = []
    for t in range(max_iter):
        grad = Q @ w - q
        # LMO: s = -r * sign(grad_j) * e_j where j = argmax |grad_i|
        j = np.argmax(np.abs(grad))
        s = np.zeros(d); s[j] = -np.sign(grad[j]) * r
        gamma = 2.0 / (t + 2)
        w = w + gamma * (s - w)
        obj_hist.append(0.5 * w @ Q @ w - q @ w)
        sparsity_hist.append(np.sum(np.abs(w) > 1e-10))
    return w, obj_hist, sparsity_hist

w_fw_sol, obj_fw, sparsity_fw = fw_l1(Q_fw_big, q_fw_big, r_ball)

# ============================================================================
# 3. Entropy mirror descent on simplex
#    min 0.5*w^T A w - b^T w   s.t. w in simplex
# ============================================================================
d_simplex = 20
A_simplex = np.random.randn(d_simplex, d_simplex)
A_simplex = A_simplex.T @ A_simplex / d_simplex + 0.3 * np.eye(d_simplex)
b_simplex = np.random.randn(d_simplex)

def mirror_entropy(A, b, eta=0.3, max_iter=200):
    """Exponentiated gradient (entropy mirror descent) on the simplex."""
    d = len(b)
    w = np.ones(d) / d
    obj_hist = []
    for _ in range(max_iter):
        grad = A @ w - b
        w = w * np.exp(-eta * grad)
        w /= w.sum()  # renormalize to simplex
        w = np.clip(w, 1e-15, None)  # numerical safety
        w /= w.sum()
        obj_hist.append(0.5 * w @ A @ w - b @ w)
    return w, obj_hist

def pgd_simplex(A, b, eta=0.1, max_iter=200):
    """Projected gradient descent (Euclidean mirror) on the simplex."""
    d = len(b)
    w = np.ones(d) / d
    obj_hist = []
    for _ in range(max_iter):
        grad = A @ w - b
        w = proj_simplex(w - eta * grad)
        obj_hist.append(0.5 * w @ A @ w - b @ w)
    return w, obj_hist

w_mirror_sol, obj_mirror = mirror_entropy(A_simplex, b_simplex)
w_pgd_sol, obj_pgd = pgd_simplex(A_simplex, b_simplex)

# ---------- Print summaries ----------
f_star_lasso = min(min(obj_admm), min(obj_ista), min(obj_fista))
print("=" * 72)
print("1. ADMM vs ISTA vs FISTA for Lasso")
print(f"   Final objectives — ADMM: {obj_admm[-1]:.6f}  ISTA: {obj_ista[-1]:.6f}  "
      f"FISTA: {obj_fista[-1]:.6f}")
print(f"   ADMM nonzeros: {np.sum(np.abs(w_admm_sol) > 1e-4)}  "
      f"(ground truth: {np.sum(np.abs(w_star) > 0)})")
print()
print("2. Frank-Wolfe on L1 ball")
print(f"   Final objective: {obj_fw[-1]:.6f}")
print(f"   ||w||_1 = {np.linalg.norm(w_fw_sol, 1):.4f} (constraint: ≤ {r_ball})")
print(f"   Nonzero coords: {np.sum(np.abs(w_fw_sol) > 1e-10)}/{d_fw} (sparse by construction)")
print()
print("3. Mirror descent vs Projected GD on simplex")
f_star_simplex = min(min(obj_mirror), min(obj_pgd))
print(f"   Final gap — entropy mirror: {obj_mirror[-1] - f_star_simplex:.2e}  "
      f"PGD: {obj_pgd[-1] - f_star_simplex:.2e}")
print(f"   Simplex check — mirror: sum={w_mirror_sol.sum():.6f} min={w_mirror_sol.min():.2e}  "
      f"PGD: sum={w_pgd_sol.sum():.6f} min={w_pgd_sol.min():.2e}")
print("=" * 72)

## 🤖 Why This Matters for AI

These three methods are not textbook curiosities — they are the computational engines
behind some of the most important large-scale ML systems:

**Federated learning and ADMM.** When training data is distributed across $K$ devices
(hospitals, phones, edge nodes), privacy constraints forbid centralizing it. ADMM's
consensus formulation $\min \sum_{i=1}^K f_i(x_i)$ s.t. $x_i = z$ lets each device
solve a local subproblem on its own data, then a server aggregates via the $z$-update
and broadcasts the updated dual variable $u$. The FedSplit and FedADMM algorithms are
direct instantiations of the ADMM framework from this notebook. Robust PCA via ADMM
also finds direct application in separating foreground from background in video
surveillance.

**Low-rank adaptation and Frank-Wolfe.** Nuclear-norm constraints
$\{W : \|W\|_* \le r\}$ enforce low-rank structure — the constraint behind matrix
completion (Netflix recommendations) and low-rank adaptation (LoRA-style parameter-
efficient fine-tuning). Projecting onto the nuclear-norm ball requires a *full SVD*
plus singular-value thresholding — $O(\min(m,n) \cdot mn)$ per iteration. Frank-Wolfe
needs only the *leading* singular vector pair (one SVD call), building rank incrementally.
For a $10{,}000 \times 10{,}000$ matrix, that's the difference between $O(10^{12})$ and
$O(10^8)$ per step.

**Online learning, boosting, and mirror descent.** The exponentiated gradient algorithm
(entropy mirror descent on the simplex) is the backbone of the **Hedge** algorithm for
online prediction with expert advice and the **multiplicative weights** method that
underlies AdaBoost. In online convex optimization, mirror descent with the KL divergence
achieves regret bounds that scale with $\log d$ rather than $\sqrt{d}$ — exponentially
better when the dimension (number of experts/features) is large. The connection to
natural gradient (`04.07`) means that mirror descent also explains why Fisher-aware
optimizers (K-FAC, natural policy gradient in RL) outperform vanilla SGD on distribution-
valued parameters.

In [ ]:
# Grand comparison: convergence of all methods on their respective problems.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# --- Panel 1: ADMM vs ISTA vs FISTA on Lasso ---
ax = axes[0]
f_best = min(min(obj_admm), min(obj_ista), min(obj_fista))
ax.semilogy(np.array(obj_admm) - f_best + 1e-15, label='ADMM', color='#4C72B0', lw=2)
ax.semilogy(np.array(obj_ista) - f_best + 1e-15, label='ISTA', color='#C44E52', lw=2, ls='--')
ax.semilogy(np.array(obj_fista) - f_best + 1e-15, label='FISTA', color='#55A868', lw=2, ls=':')
ax.set_xlabel('iteration'); ax.set_ylabel('f(w) - f* (log)')
ax.set_title('Lasso: ADMM vs ISTA vs FISTA')
ax.legend(fontsize=9)

# --- Panel 2: Frank-Wolfe convergence + sparsity ---
ax = axes[1]
color1, color2 = '#4C72B0', '#C44E52'
ax.semilogy(np.array(obj_fw) - min(obj_fw) + 1e-15, color=color1, lw=2, label='f(w) - f*')
ax.set_xlabel('iteration'); ax.set_ylabel('objective gap (log)', color=color1)
ax.tick_params(axis='y', labelcolor=color1)
ax2_twin = ax.twinx()
ax2_twin.plot(sparsity_fw, color=color2, lw=2, ls='--', label='nonzero coords')
ax2_twin.set_ylabel('# nonzero coordinates', color=color2)
ax2_twin.tick_params(axis='y', labelcolor=color2)
ax.set_title('Frank-Wolfe: convergence + sparsity\n(sparse iterates by construction)')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='center right')

# --- Panel 3: Mirror descent vs PGD ---
ax = axes[2]
f_best_s = min(min(obj_mirror), min(obj_pgd))
ax.semilogy(np.array(obj_mirror) - f_best_s + 1e-15, label='Entropy mirror (EG)',
            color='#55A868', lw=2)
ax.semilogy(np.array(obj_pgd) - f_best_s + 1e-15, label='Projected GD (Euclidean)',
            color='#4C72B0', lw=2, ls='--')
ax.set_xlabel('iteration'); ax.set_ylabel('f(w) - f* (log)')
ax.set_title('Simplex optimization:\nentropy mirror vs Euclidean PGD')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# ---------- Summary statistics ----------
print("\nConvergence summary (iterations to reach within 1e-3 of best):")
for name, obj_list in [("ADMM", obj_admm), ("ISTA", obj_ista), ("FISTA", obj_fista)]:
    gaps = np.array(obj_list) - f_best
    reached = np.where(gaps < 1e-3)[0]
    iters = reached[0] if len(reached) > 0 else ">200"
    print(f"  {name:6s}: {iters} iterations")

print(f"\nFrank-Wolfe: {sparsity_fw[-1]}/{d_fw} nonzero coordinates after {len(obj_fw)} iters")
print(f"  (each FW step adds at most 1 vertex → sparsity bounded by T)")

print(f"\nSimplex methods final gap:")
print(f"  Entropy mirror: {obj_mirror[-1] - f_best_s:.2e}")
print(f"  Projected GD:   {obj_pgd[-1] - f_best_s:.2e}")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Consider ADMM for 2D Lasso with $X = I_2$, $y = (3, 0.2)^\top$, $\lambda = 0.5$,
   $\rho = 1$. Starting from $w^0 = z^0 = u^0 = \mathbf{0}$, execute one full ADMM
   iteration by hand: compute $w^1$ (solve the ridge system), $z^1$ (soft-threshold),
   and $u^1$ (dual update). Verify that the primal residual $\|w^1 - z^1\|$ is smaller
   than $\|w^0 - z^0\| = 0$... wait, both start at zero! Explain why the first iteration
   *creates* a nonzero residual and how subsequent iterations drive it back down.

<details>
<summary>💡 Solution</summary>

With $X = I_2$, $X^\top X = I_2$. The $x$-update solves $(I + \rho I)w = X^\top y +
\rho(z^0 - u^0) = (3, 0.2)^\top$, giving $w^1 = (3, 0.2)^\top / 2 = (1.5, 0.1)^\top$.
The $z$-update: $z^1 = \mathcal{S}_{\lambda/\rho}(w^1 + u^0) = \mathcal{S}_{0.5}((1.5,
0.1)^\top) = (1.0, 0)^\top$ (the second coordinate is within the dead zone $[-0.5, 0.5]$
and gets thresholded to exactly 0). The dual update: $u^1 = u^0 + w^1 - z^1 = (0.5,
0.1)^\top$. The primal residual $\|w^1 - z^1\| = \|(0.5, 0.1)\| \approx 0.51$ — nonzero
because the $x$-update and $z$-update solve *different* objectives (one sees the data fit,
the other sees the sparsity penalty), so they don't immediately agree. The dual variable
$u^1$ records this disagreement, and the next $x$-update will account for it, progressively
shrinking the residual.

</details>

### 💭 UNDERSTAND
2. Frank-Wolfe converges at $O(1/T)$ — the same rate as projected GD. Why would you ever
   prefer it? Give a concrete example where the per-iteration cost of projected GD is
   prohibitive but Frank-Wolfe's LMO is cheap.

<details>
<summary>💡 Solution</summary>

Consider the nuclear-norm ball $\{W : \|W\|_* \le r\}$ for an $m \times n$ matrix.
**Projected GD** requires projecting onto this set, which needs a *full SVD* ($O(\min(m,n)
\cdot mn)$) plus singular-value soft-thresholding — for a $10{,}000 \times 10{,}000$ matrix,
that's $O(10^{12})$ operations *per iteration*. **Frank-Wolfe's LMO** only needs the
*leading* singular vector pair (a single call to a power-method or Lanczos routine),
costing $O(mn)$ — a factor of $\min(m,n)$ cheaper. The LMO returns $r\,u_1 v_1^\top$
(rank-1), so Frank-Wolfe builds a low-rank solution incrementally, one rank at a time.
The $O(1/T)$ rate is the same, but $T = 100$ Frank-Wolfe iterations may be faster in
wall-clock time than $T = 100$ projected-GD iterations by orders of magnitude.

</details>

### ✏️ DERIVE
3. Show that mirror descent with the Euclidean mirror map $\phi(w) = \frac{1}{2}\|w\|_2^2$
   recovers projected gradient descent. Start from the general mirror descent update
   $w_{t+1} = \arg\min_{w \in C}\{\langle g_t, w \rangle + \frac{1}{\eta} D_\phi(w, w_t)\}$,
   compute $D_\phi(w, w_t)$ for this $\phi$, simplify the optimization problem, and identify
   the result as the projected gradient step $\text{Proj}_C(w_t - \eta g_t)$.

<details>
<summary>💡 Solution outline</summary>

For $\phi(w) = \frac{1}{2}\|w\|_2^2$: $\nabla\phi(w) = w$, so $D_\phi(w, v) = \phi(w) -
\phi(v) - \nabla\phi(v)^\top(w - v) = \frac{1}{2}\|w\|^2 - \frac{1}{2}\|v\|^2 - v^\top
(w - v) = \frac{1}{2}\|w - v\|^2$. Substituting: $w_{t+1} = \arg\min_{w \in C}\{g_t^\top
w + \frac{1}{2\eta}\|w - w_t\|^2\}$. Complete the square: $g_t^\top w + \frac{1}{2\eta}
\|w - w_t\|^2 = \frac{1}{2\eta}\|w - (w_t - \eta g_t)\|^2 + \text{const}$. Minimizing
this over $C$ is exactly $\text{Proj}_C(w_t - \eta g_t)$ — the definition of projected
gradient descent. QED. The entropy mirror map $\phi(w) = \sum w_j \log w_j$ gives
$D_\phi(w, v) = \text{KL}(w \| v)$ and a fundamentally different update — the geometry
of the Bregman divergence determines the algorithm.

</details>

### 🐍 IMPLEMENT
4. Implement ADMM for **robust PCA**: given $M = L^* + S^*$ where $L^*$ is low-rank and
   $S^*$ is sparse, solve $\min_{L, S} \|L\|_* + \lambda\|S\|_1$ subject to $L + S = M$.
   The $L$-update requires singular-value soft-thresholding (threshold the singular values
   of $M - S^k + U^k$), and the $S$-update is elementwise soft-thresholding. Generate a
   $50 \times 50$ matrix with rank 3 plus 5% sparse corruption, and recover $L$ and $S$.

<details>
<summary>✅ How to know you're right</summary>

The recovered $L$ should have numerical rank close to 3 (singular values drop sharply
after the 3rd), and $\|L_{\text{recovered}} - L^*\|_F / \|L^*\|_F$ should be below 0.1.
The recovered $S$ should be sparse with nonzeros concentrated at the same locations as
$S^*$, with $\|S_{\text{recovered}} - S^*\|_F / \|S^*\|_F < 0.2$. The constraint
residual $\|L + S - M\|_F$ should be below $10^{-4}$ after 200 iterations with $\rho = 1$.

</details>

### 🤖 APPLY
5. Compare ADMM, ISTA, and coordinate descent (from `04.11`) on a large sparse regression
   problem: $n = 500$ samples, $d = 10{,}000$ features, 20 true nonzeros. Use
   $\lambda = 0.1 \cdot \lambda_{\max}$ (where $\lambda_{\max} = \|X^\top y\|_\infty$ is
   the smallest $\lambda$ that zeros out the entire solution). Compare: (a) iterations to
   reach $10^{-4}$ suboptimality, (b) wall-clock time, (c) memory usage. Which method
   wins on each metric, and why?

<details>
<summary>✅ What you should observe</summary>

**ADMM** converges in relatively few iterations but each iteration is expensive (the
$x$-update requires solving a $d \times d$ linear system — $O(d^3)$ via Cholesky or
$O(d^2)$ via CG). **ISTA** is the cheapest per iteration (just a matrix-vector product
+ soft-thresholding) but converges slowly ($O(1/T)$). **Coordinate descent** has the
lowest per-coordinate cost and exploits sparsity aggressively — it typically wins on
wall-clock time for large $d$ when the solution is sparse, which is why sklearn uses it.
Memory: ADMM stores $w, z, u$ plus the Cholesky factor ($O(d^2)$); ISTA and coordinate
descent need only $O(nd)$ for the data matrix.

</details>

### 🚀 CHALLENGE
6. Prove the Frank-Wolfe convergence rate. Starting from the curvature constant
   $C_f = \sup_{x, s \in C,\, \gamma \in [0,1]} \frac{2}{\gamma^2}[f(x + \gamma(s - x))
   - f(x) - \gamma\langle \nabla f(x), s - x \rangle]$, show by induction that with
   $\gamma_t = \frac{2}{t+2}$:

   $$f(w_T) - f^* \le \frac{2C_f}{T + 2}$$

   *Hint:* At each step, bound $f(w_{t+1}) - f(w_t)$ using the curvature constant, then
   use $\langle \nabla f(w_t), s_t - w_t \rangle \le \langle \nabla f(w_t), w^* - w_t
   \rangle \le f^* - f(w_t)$ (convexity + the LMO definition), and telescope.

<details>
<summary>🔍 Proof sketch</summary>

Define $h_t = f(w_t) - f^*$. From the curvature constant definition with $x = w_t$,
$s = s_t$: $f(w_{t+1}) \le f(w_t) + \gamma_t \langle \nabla f(w_t), s_t - w_t \rangle
+ \frac{\gamma_t^2 C_f}{2}$. The LMO guarantees $\langle \nabla f(w_t), s_t \rangle
\le \langle \nabla f(w_t), w^* \rangle$, so $\langle \nabla f(w_t), s_t - w_t \rangle
\le \langle \nabla f(w_t), w^* - w_t \rangle \le f^* - f(w_t) = -h_t$ (convexity).
Substituting: $h_{t+1} \le h_t(1 - \gamma_t) + \frac{\gamma_t^2 C_f}{2}$. Now induct:
assume $h_t \le \frac{2C_f}{t+2}$. With $\gamma_t = \frac{2}{t+2}$: $h_{t+1} \le
\frac{2C_f}{t+2} \cdot \frac{t}{t+2} + \frac{2C_f}{(t+2)^2} = \frac{2C_f(t+1)}{(t+2)^2}
\le \frac{2C_f}{t+3}$, completing the induction. The base case $h_0 \le C_f = \frac{2C_f}
{0+2}$ holds by definition of $C_f$ with $\gamma = 1$.

</details>

---

## 📚 Further Reading
- Boyd et al., [*Distributed Optimization and Statistical Learning via ADMM*](https://stanford.edu/~boyd/papers/admm_distr_stats.html) (2011) — the definitive ADMM monograph
- Jaggi, ["Revisiting Frank-Wolfe: Projection-Free Sparse Convex Optimization"](http://proceedings.mlr.press/v28/jaggi13.html) (ICML 2013)
- Bubeck, ["Convex Optimization: Algorithms and Complexity"](https://arxiv.org/abs/1405.4980) (2015), §§4–5 (mirror descent, Frank-Wolfe)
- Parikh & Boyd, [*Proximal Algorithms*](https://web.stanford.edu/~boyd/papers/prox_algs.html) (2014) — ADMM as a proximal method
- Johnson & Zhang, ["Accelerating Stochastic Gradient via SVRG"](https://papers.nips.cc/paper/2013/hash/ac1dd209cbcc5e5d1c6e28598e8cbbe8-Abstract.html) (NeurIPS 2013)
- Shalev-Shwartz, ["Online Learning and Online Convex Optimization"](https://www.cs.huji.ac.il/~shais/papers/OLsurvey.pdf) (2012) — mirror descent in online learning

---

*End of Module 04*